In [4]:
import mlflow
import pandas as pd
import numpy as np
import optuna
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [5]:
import mlflow

# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_experiment("Hyperparameter Wine Quality Tuning Experiment")

2026/01/18 12:34:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/01/18 12:34:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/01/18 12:34:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/01/18 12:34:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/01/18 12:34:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/01/18 12:34:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/01/18 12:34:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/18 12:34:51 INFO mlflow.store.db.utils: Updating database tables
2026/01/18 12:34:51 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/18 12:34:51 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/18 12:34:51 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/18 12:34:51 INFO alembic.runtime

<Experiment: artifact_location='file:///c:/Users/jeroen.vander.putten/Development/Learning/Python/github_portfolio_jeroen/project_MLflow/mlruns/4', creation_time=1768736091216, experiment_id='4', last_update_time=1768736091216, lifecycle_stage='active', name='Hyperparameter Wine Quality Tuning Experiment', tags={}>

In [10]:
# Load and preprocess the data
data = pd.read_csv("WineQt.csv")

# Rename columns to replace spaces with underscores
data.rename(columns=lambda x: x.replace(' ', '_'), inplace=True)

# Create a binary target variable for high quality wines
high_quality = (data.quality >= 7).astype(int)
data['high_quality'] = high_quality

#
X = data.drop(["quality", "high_quality", "Id"], axis=1)
y = data["high_quality"]

# Split out the data
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=123)


In [15]:

def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:

        # Hyperparameters and search space
        max_depth = trial.suggest_int("max_depth", 2, 32) # number of trees
        n_estimators = trial.suggest_int("n_estimators", 50, 300, step=10) # depth of each tree
        max_features = trial.suggest_float("max_features", 0.2, 0.8) # number of features to consider at each split
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 2, 10) # minimum samples required at each leaf node

        params = {
            "max_depth": max_depth,
            "n_estimators": n_estimators,
            "max_features": max_features,
            "min_samples_leaf": min_samples_leaf,
            "random_state": 42,
            "n_jobs": 1
}

        # Log current trial's parameters
        mlflow.log_params(params) 

        # Train and evaluate the model
        rf = sklearn.ensemble.RandomForestClassifier(**params) # Instantiate the model
        rf.fit(X_train, y_train) # Fit to training data
        y_score = rf.predict_proba(X_test)[:, 1] # Get predicted probabilities
        roc_auc = sklearn.metrics.roc_auc_score(y_test, y_score) # Calculate ROC AUC

        # Log current trial's error metric
        mlflow.log_metrics({"roc_auc_score": roc_auc})

        # Log the model file
        mlflow.sklearn.log_model(rf, name="random_forest_model")

        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        return roc_auc

In [16]:
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="study") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.RandomSampler(seed=123))
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_roc_auc_score": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-01-18 12:55:15,341] A new study created in memory with name: no-name-6ee4879b-5a22-40ce-a186-ff9006ce32b5
2026/01/18 12:55:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-01-18 12:55:18,075] Trial 0 finished with value: 0.8613861386138613 and parameters: {'max_depth': 23, 'n_estimators': 120, 'max_features': 0.33611087213852187, 'min_samples_leaf': 6}. Best is trial 0 with value: 0.8613861386138613.
2026/01/18 12:55:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-01-18 12:55:20,744] Trial 1 finished with value: 0.8561881188118812 and parameters: {'max_depth': 24, 'n_estimators': 160, 'max_features': 0.7884585190307694, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.8613861386138613.
2026/01/18 12:55:23 WARNING mlflow.utils.environment: